In [1]:
!nvidia-smi
!pip install ultralytics --quiet
!pip install opencv-python-headless --quiet

import ultralytics
ultralytics.checks()
from google.colab import drive
drive.mount('/content/drive')

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (8 CPUs, 51.0 GB RAM, 42.9/235.7 GB disk)
Mounted at /content/drive


In [13]:
import wandb
from google.colab import userdata
import os
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
wandb.init(project="Lab_2_Tests", entity="ertveh-4-lule-university-of-technology")

In [3]:
import os
import zipfile

ZIP_PATH = '/content/drive/MyDrive/Training_data_10p.zip'
EXTRACT_DIR = '/content/dataset'

os.makedirs(EXTRACT_DIR, exist_ok=True)

print('Extracting...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Done.')

# Verify structure
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:3]:
            print(f'{indent}  {f}')

Extracting...
Done.
dataset/
  Training_data_10p/
    data_10_full.yaml
    dataset/
      labels/
        train/
        val/
      images/
        train/
        val/


In [7]:
import yaml

dataset_config = {
    'path': '/content/dataset/Training_data_10p/dataset',
    'train': 'images/train',
    'val':   'images/val',
    'nc': 1,
    'names': ['car']
}

yaml_path = '/content/dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(open(yaml_path).read())

names:
- car
nc: 1
path: /content/dataset/Training_data_10p/dataset
train: images/train
val: images/val



In [8]:
import cv2
import matplotlib.pyplot as plt
import random
import glob

def draw_yolo_boxes(img_path, label_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                cls, cx, cy, bw, bh = map(float, line.strip().split())
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
    return img

image_files = glob.glob('/content/dataset/Training_data_10p/dataset/images/train/**/*.jpg', recursive=True) + \
              glob.glob('/content/dataset/Training_data_10p/dataset/images/train/**/*.png', recursive=True)

samples = random.sample(image_files, min(6, len(image_files)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img_path in zip(axes.flatten(), samples):
    label_path = img_path.replace('images', 'labels').rsplit('.', 1)[0] + '.txt'
    img = draw_yolo_boxes(img_path, label_path)
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Total training images: {len(image_files)}')

Output hidden; open in https://colab.research.google.com to view.

In [14]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')  # pretrained on COCO

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=1280,
    batch=4,
    device=0,
    workers=2,
    project='/content/runs',
    name='yolo11m_nvd_10p_1280',
    exist_ok=True,
    pretrained=True,
    freeze=None,
    patience=20,
    save=True,
    plots=True,
    cache='disk',
    amp=True
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11m_nvd_10p_1280, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7884dd9c7f60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7884dd9c7f60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

      13/50      9.08G      1.013     0.5837     0.8205         23       1280: 100% ━━━━━━━━━━━━ 1665/1665 1.6it/s 16:58
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 316/316 4.2it/s 1:16
                   all       2523       4156      0.669      0.452      0.494      0.293

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      14/50      9.09G     0.9902     0.5574     0.8184         17       1280: 87% ━━━━━━━━━━── 1452/1665 1.6it/s 14:50<2:11

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7884dd9c7f60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7884dd9c7f60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

      14/50      9.09G     0.9878     0.5567     0.8178          5       1280: 100% ━━━━━━━━━━━━ 1665/1665 1.6it/s 16:60
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 316/316 4.1it/s 1:17
                   all       2523       4156      0.683      0.478      0.512      0.285

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      15/50      9.09G     0.9744     0.5631     0.8172         12       1280: 100% ━━━━━━━━━━━━ 1665/1665 1.6it/s 16:59
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 316/316 4.1it/s 1:16
                   all       2523       4156      0.662      0.443      0.477      0.263

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      16/50      9.06G     0.9601     0.5496     0.8128          7       1280: 100% ━━━━━━━━━━━━ 1665/1665 1.6it/s 17:04
                 Class     Images  Instance

KeyboardInterrupt: 

In [15]:
best_model = YOLO('/content/runs/yolo11m_nvd_10p_1280/weights/best.pt')

metrics = best_model.val(data=yaml_path, imgsz=1280, device=0)

print(f"mAP@50:     {metrics.box.map50:.4f}")
print(f"mAP@50-95:  {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 91.0±30.0 MB/s, size: 515.1 KB)
val: Scanning /content/dataset/Training_data_10p/dataset/labels/val.cache... 1203 images, 1323 backgrounds, 3 corrupt: 100% ━━━━━━━━━━━━ 2526/2526 529.7Mit/s 0.0s
val: /content/dataset/Training_data_10p/dataset/images/val/frame_1944.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0105      1.0112]
val: /content/dataset/Training_data_10p/dataset/images/val/frame_2238.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0119]
val: /content/dataset/Training_data_10p/dataset/images/val/frame_381.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0116]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95

In [16]:
import shutil, os

DRIVE_OUTPUT = '/content/drive/MyDrive/D7047E Project Datasets/yolo11_results'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

shutil.copy('/content/runs/yolo11m_nvd_10p_1280/weights/best.pt',
            os.path.join(DRIVE_OUTPUT, 'yolo11m_10p_best.pt'))
shutil.copy('/content/runs/yolo11m_nvd_10p_1280/weights/last.pt',
            os.path.join(DRIVE_OUTPUT, 'yolo11m_10p_last.pt'))
shutil.copy('/content/runs/yolo11m_nvd_10p_1280/results.csv',
            os.path.join(DRIVE_OUTPUT, 'yolo11m_10p_results.csv'))

print('Saved.')

Saved.
